In [0]:
from pyspark.sql.functions import col, lag, datediff
from pyspark.sql.window import Window

telemetry = spark.table("workspace.default.silver_telemetry")
maint = spark.table("workspace.default.silver_maint")

print("Telemetry Schema:")
telemetry.printSchema()

print("Maintenance Schema:")
maint.printSchema()

Telemetry Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)

Maintenance Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- comp: string (nullable = true)



In [0]:
from pyspark.sql.functions import max

telemetry_with_maint = (
    telemetry.alias("t")
    .join(
        maint.alias("m"),
        (col("t.machineID") == col("m.machineID")) &
        (col("m.datetime") <= col("t.datetime")),
        "left"
    )
    .groupBy(
        "t.datetime",
        "t.machineID",
        "t.volt",
        "t.rotate",
        "t.pressure",
        "t.vibration"
    )
    .agg(
        max("m.datetime").alias("last_maintenance")
    )
)

telemetry_with_maint.show(10, truncate=False)

+-------------------+---------+----------------+----------------+----------------+----------------+-------------------+
|datetime           |machineID|volt            |rotate          |pressure        |vibration       |last_maintenance   |
+-------------------+---------+----------------+----------------+----------------+----------------+-------------------+
|2015-07-23 10:00:00|1        |151.102005103031|423.622014828634|101.493063423781|45.3727305991408|2015-07-19 06:00:00|
|2015-08-23 02:00:00|1        |172.943939812115|444.399901834319|94.6415045707178|42.5484301033552|2015-08-03 06:00:00|
|2015-09-09 11:00:00|1        |164.163778702088|389.117764502401|101.968608486417|41.1942172959639|2015-09-02 06:00:00|
|2015-12-21 03:00:00|2        |170.845900036235|345.818009384898|108.460066321634|39.2145358994983|2015-12-14 06:00:00|
|2015-05-28 19:00:00|3        |159.73548873961 |402.723098683041|113.296186271656|36.2596278992153|2015-05-22 06:00:00|
|2015-06-25 13:00:00|3        |175.39498

In [0]:
from pyspark.sql.functions import datediff

telemetry_with_maint = (
    telemetry_with_maint
    .withColumn(
        "days_since_maintenance",
        datediff(col("datetime"), col("last_maintenance"))
    )
)

telemetry_with_maint.show(10, truncate=False)

+-------------------+---------+----------------+----------------+----------------+----------------+-------------------+----------------------+
|datetime           |machineID|volt            |rotate          |pressure        |vibration       |last_maintenance   |days_since_maintenance|
+-------------------+---------+----------------+----------------+----------------+----------------+-------------------+----------------------+
|2015-07-23 10:00:00|1        |151.102005103031|423.622014828634|101.493063423781|45.3727305991408|2015-07-19 06:00:00|4                     |
|2015-08-23 02:00:00|1        |172.943939812115|444.399901834319|94.6415045707178|42.5484301033552|2015-08-03 06:00:00|20                    |
|2015-09-09 11:00:00|1        |164.163778702088|389.117764502401|101.968608486417|41.1942172959639|2015-09-02 06:00:00|7                     |
|2015-12-21 03:00:00|2        |170.845900036235|345.818009384898|108.460066321634|39.2145358994983|2015-12-14 06:00:00|7                     |

In [0]:
from pyspark.sql.functions import avg, stddev

maintenance_stability = (
    telemetry_with_maint
    .groupBy("days_since_maintenance")
    .agg(
        avg("volt").alias("avg_voltage"),
        stddev("volt").alias("voltage_variability"),
        avg("rotate").alias("avg_rotation"),
        stddev("rotate").alias("rotation_variability"),
        avg("pressure").alias("avg_pressure"),
        stddev("pressure").alias("pressure_variability"),
        avg("vibration").alias("avg_vibration"),
        stddev("vibration").alias("vibration_variability")
    )
    .orderBy("days_since_maintenance")
)

maintenance_stability.show(50, truncate=False)

+----------------------+------------------+-------------------+------------------+--------------------+------------------+--------------------+------------------+---------------------+
|days_since_maintenance|avg_voltage       |voltage_variability|avg_rotation      |rotation_variability|avg_pressure      |pressure_variability|avg_vibration     |vibration_variability|
+----------------------+------------------+-------------------+------------------+--------------------+------------------+--------------------+------------------+---------------------+
|0                     |170.58334020336187|15.388039566999671 |447.62144044334553|51.75523840090864   |100.70210056711358|10.930350743529385  |40.32419684856547 |5.279163729224901    |
|1                     |170.4898325105512 |15.333612576770323 |447.4621256426375 |51.98076002466368   |100.6087476122968 |10.824473449134954  |40.22085463552239 |5.213027499662278    |
|2                     |170.57631013130788|15.313038555211106 |447.59286775

In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.gold_q3_maintenance_stability"

if not spark.catalog.tableExists(target_table):

    maintenance_stability.limit(0).write \
        .format("delta") \
        .saveAsTable(target_table)

    print("Gold Q3 table created")

target = DeltaTable.forName(spark, target_table)

target.alias("t").merge(
    maintenance_stability.alias("s"),
    "t.days_since_maintenance = s.days_since_maintenance"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Q3 maintenance stability table updated")

target.toDF().printSchema()

Gold Q3 table created
Gold Q3 maintenance stability table updated
root
 |-- days_since_maintenance: integer (nullable = true)
 |-- avg_voltage: double (nullable = true)
 |-- voltage_variability: double (nullable = true)
 |-- avg_rotation: double (nullable = true)
 |-- rotation_variability: double (nullable = true)
 |-- avg_pressure: double (nullable = true)
 |-- pressure_variability: double (nullable = true)
 |-- avg_vibration: double (nullable = true)
 |-- vibration_variability: double (nullable = true)



In [0]:
display(maintenance_stability)

days_since_maintenance,avg_voltage,voltage_variability,avg_rotation,rotation_variability,avg_pressure,pressure_variability,avg_vibration,vibration_variability
0,170.58334020336187,15.388039566999671,447.62144044334553,51.75523840090864,100.70210056711358,10.930350743529385,40.32419684856547,5.279163729224901
1,170.4898325105512,15.333612576770323,447.4621256426375,51.98076002466368,100.6087476122968,10.824473449134954,40.22085463552239,5.213027499662278
2,170.57631013130788,15.313038555211106,447.59286775443627,51.60946080923649,100.76712565909122,10.93228497083024,40.25294807474026,5.281574513089666
3,170.5097000654589,15.404280127771226,447.8005181942246,51.89424874329279,100.69348199103173,10.84902345208302,40.26582618638411,5.278479543124175
4,170.81483571541045,15.46283210510513,447.6774359070225,51.656022087379434,100.52409358745922,10.722051537343273,40.32627413084535,5.314723091961636
5,170.65076998441816,15.385770580432922,448.5973725339378,51.42698434893376,100.625130090318,10.733478655763367,40.40731916334801,5.361646028715402
6,170.57454547587483,15.43034868895094,447.5442451900517,51.66228919193748,100.7003887537399,10.852673880548128,40.36196308951121,5.351604704131317
7,170.6030349284215,15.460366385039617,447.90692062296506,51.9162962431706,100.59506742350125,10.722609310472746,40.291765594774134,5.261915517125762
8,170.5244498288317,15.325517258774893,447.859553491475,51.57683389067179,100.62785471636708,10.65584164011437,40.2672662250862,5.2663850878348715
9,170.57614935867258,15.348733234833354,447.97870282026844,51.9709200233073,100.6740444915405,10.873026469723806,40.24460218161578,5.247824280762987


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.